In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = Path("../results")

/Users/ben/mambaforge/lib/python3.12/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.3)
  from scipy.stats import gaussian_kde


In [2]:
records = []
for json_file in RESULTS_DIR.rglob("*.json"):
    data = json.loads(json_file.read_text())
    for item in data:
        records.append({
            "model": item["model"],
            "task": item["task"],
            "condition": item["condition"],
            "query": item["stimulus"]["query"],
            "expected": item["stimulus"]["expected"],
            "few_shot_examples": item["stimulus"]["few_shot_examples"],
            "metadata": item["stimulus"]["metadata"],
            "response_text": item["response"]["text"],
            "token_logprobs": item["response"]["token_logprobs"],
            "correct": item["score"]["correct"],
            "logprob_correct": item["score"]["logprob_correct"],
            "timestamp": item["timestamp"],
        })

df = pd.DataFrame(records)
print(f"Loaded {len(df)} trials from {df['model'].nunique() if len(df) > 0 else 0} models")
df.head()

Loaded 0 trials from 0 models


""


In [3]:
if df.empty:
    print("No results loaded yet.")
else:
    accuracy = (
        df.groupby(["model", "task", "condition"])["correct"]
        .mean()
        .mul(100)
        .round(1)
        .unstack("condition")
        .reset_index()
    )
    accuracy.columns.name = None
    print(accuracy.to_string(index=False))

No results loaded yet.


In [4]:
if df.empty:
    print("No results loaded yet.")
else:
    pivot = df.groupby(["model", "task"])["correct"].mean().unstack("task")
    plt.figure(figsize=(8, max(3, len(pivot) * 0.6)))
    sns.heatmap(pivot, annot=True, fmt=".0%", cmap="YlGn", vmin=0, vmax=1)
    plt.title("Accuracy by model and task")
    plt.tight_layout()
    plt.show()

No results loaded yet.


In [5]:
if df.empty or df["logprob_correct"].isna().all():
    print("No log prob data available yet.")
else:
    lp_df = df.dropna(subset=["logprob_correct"])
    g = sns.FacetGrid(lp_df, col="task", row="condition", height=3, aspect=1.5)
    g.map_dataframe(sns.boxplot, x="model", y="logprob_correct")
    g.set_axis_labels("Model", "Log P(correct)")
    g.set_titles("{row_name} | {col_name}")
    plt.tight_layout()
    plt.show()

No log prob data available yet.


In [6]:
if df.empty:
    print("No results loaded yet.")
else:
    df["rule"] = df["metadata"].apply(lambda m: m.get("rule") or m.get("pattern", "unknown"))
    rule_acc = (
        df.groupby(["task", "rule", "condition"])["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
        .reset_index()
    )
    rule_acc["accuracy"] = rule_acc["accuracy"].mul(100).round(1)
    print(rule_acc.to_string(index=False))

No results loaded yet.
